# Parameter-optimization with Optuna

## 事前準備

In [1]:
import torch

# GPUが使えるか確認してデバイスを設定
# NOTE: `x = x.to(device) ` とすることで対象のデバイスに切り替え可能
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from typing import Callable

import random
import torch
import numpy as np


# シード値の固定
# NOTE: 戻り値はDataLoaderのシード固定に使用する。　
# 　　　　　　　　　　　　(ex) loader = DataLoader(..., worker_init_fn=seed_worker, generator=generator)
def fixing_seed(seed: int=42) -> tuple[torch.Generator, Callable]:
    # Python のシード固定
    random.seed(seed)
    # Numpy のシード固定
    np.random.seed(seed)
    # PyTorch のシード固定
    torch.manual_seed(seed)
    # CUDA の再現性確保の設定
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    # DataLoader 各ワーカー用の初期化関数
    def seed_worker_fn(worker_id: int) -> None:
        # 各 worker で Python / NumPy も固定
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    # DataLoader の乱数源
    generator = torch.Generator()
    generator.manual_seed(seed)

    return generator, seed_worker_fn

In [3]:
SEED = 24

torch_generator, seed_worker_fn = fixing_seed(SEED)

## CNN

In [4]:
import optuna
import torch
import torchinfo
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.datasets import CIFAR10
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt

### DataLoader設定

In [5]:
# transformを準備
affine = transforms.RandomAffine((-30, 30), scale=(0.8, 1.2))
flip = transforms.RandomHorizontalFlip(p=0.5)
normalize = transforms.Normalize((0.0, 0.0, 0.0), (1.0, 1.0, 1.0))  # 平均0、標準偏差1

transform_train = transforms.Compose([
    affine,
    flip,
    transforms.ToTensor(),
    normalize
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    normalize
])

In [6]:
# DataLoader作成
cifar10_train = CIFAR10(root='../../.cache/data', train=True, download=True, transform=transform_train)
cifar10_test = CIFAR10(root='../../.cache/data', train=False, download=True, transform=transform_test)
cifar10_classes = cifar10_train.classes

In [7]:
# DataLoaderの設定
BATCH_SIZE = 128

train_loader = DataLoader(cifar10_train, batch_size=BATCH_SIZE, shuffle=True, generator=torch_generator, worker_init_fn=seed_worker_fn)
test_loader = DataLoader(cifar10_test, batch_size=BATCH_SIZE, shuffle=False, generator=torch_generator, worker_init_fn=seed_worker_fn)

In [8]:
len(cifar10_train), len(cifar10_test)

(50000, 10000)

### モデル構築

`dropout_prob`, `activation_func` を自動チューニングするため、コンストラクタで設定できるようにモデルクラスを作成する。

In [9]:
class Net(nn.Module):
    def __init__(self, n_classes: int,
                       dropout_prob: float = 0.5,
                       activation_func: nn.Module = nn.ReLU()):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 8, 5)         # 入力チャネル、出力チャネル、フィルタ数
        self.active = activation_func
        self.pool = nn.MaxPool2d(2, 2)          # 領域のサイズ、領域の間隔
        self.conv2 = nn.Conv2d(8, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 256)
        self.dropout = nn.Dropout(dropout_prob)          # ドロップアウト率
        self.fc2 = nn.Linear(256, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.active(self.conv1(x))
        x = self.pool(x)
        x = self.active(self.conv2(x))
        x = self.pool(x)
        x = x.view(-1, 16 * 5 * 5)
        x = self.active(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [10]:
net = Net(len(cifar10_classes))

### 訓練 & 検証用関数を作成

自動パラメータチューニングで作成するoptunaの目的関数で使用するため、訓練 & 検証用の関数を作成する。

In [11]:
def train(net: nn.Module, train_loader: DataLoader, optimizer: optim.Optimizer, criterion: nn.Module = nn.CrossEntropyLoss(), verbose: bool = True) -> float:
    net.train()
    loss_train = 0.0

    # verbose がTrueの場合はプログレスバーを表示する
    iterator = tqdm(train_loader) if verbose else train_loader

    for (x, t) in iterator:
        x, t = x.to(device), t.to(device)
        y = net(x)

        loss = criterion(y, t)
        loss_train += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    loss_train /= len(train_loader)
    return loss_train

In [12]:
def test(net: nn.Module, test_loader: DataLoader, criterion: nn.Module = nn.CrossEntropyLoss(), verbose: bool = True) -> float:
    net.eval()
    loss_test = 0.0

    # verbose がTrueの場合はプログレスバーを表示する
    iterator = tqdm(train_loader) if verbose else train_loader

    for (x, t) in iterator:
        x, t = x.to(device), t.to(device)
        y = net(x)

        loss = criterion(y, t)
        loss_test += loss.item()

    loss_test /= len(test_loader)
    return loss_test

### 自動チューニング設定(optuna)

今回は以下のパラメータをチューニング対象とする。

- 最適化アルゴリズム
- 活性化関数
- ドロップアウト率

#### Optimizer

In [13]:
def get_adam_optimizer(trial: optuna.trial.Trial, model: nn.Module) -> optim.Optimizer:
    lr = trial.suggest_float('adam_lr', 1e-5, 1e-1, log=True)
    weight_decay = trial.suggest_float('adam_weight_decay', 1e-10, 1e-3)
    optimizer = optim.Adam(model.parameters(),
                           lr=lr,
                           weight_decay=weight_decay)
    return optimizer

def get_momentum_sgd_optimizer(trial: optuna.trial.Trial, model: nn.Module) -> optim.Optimizer:
    lr = trial.suggest_float('momentum_sgd_lr', 1e-5, 1e-1, log=True)
    weight_decay = trial.suggest_float('momentum_sgd_weight_decay', 1e-10, 1e-3, log=True)
    optimizer = optim.SGD(model.parameters(),
                          lr=lr,
                          momentum=0.9,
                          weight_decay=weight_decay)
    return optimizer

def get_rms_prob_optimizer(trial: optuna.trial.Trial, model: nn.Module) -> optim.Optimizer:
    lr = trial.suggest_float('rms_prob_lr', 1e-5, 1e-1, log=True)
    optimizer = optim.RMSprop(model.parameters(), lr=lr)
    return optimizer

In [14]:
def get_optimizer(trial: optuna.trial.Trial, model: nn.Module) -> optim.Optimizer:
    optimizer_names = ['Adam', 'MomentumSGD', 'rmsprop']
    optimizer_name = trial.suggest_categorical('optimizer', optimizer_names)

    if optimizer_name == 'Adam':
        optimizer = get_adam_optimizer(trial, model)
    elif optimizer_name == 'MomentumSGD':
        optimizer = get_momentum_sgd_optimizer(trial, model)
    else:
        optimizer = get_rms_prob_optimizer(trial, model)

    return optimizer

#### 活性化関数

In [15]:
def get_activation(trial: optuna.trial.Trial) -> nn.Module:
    activation_names = ['ReLU', 'Tanh']
    activation_name = trial.suggest_categorical('activation', activation_names)

    if activation_name == 'ReLU':
        activation = nn.ReLU()
    else:
        activation = nn.Tanh()
    return activation

#### 目的関数(クラス)の定義

評価時の誤り率を最小化するような目的関数を作成する。

In [16]:
class Objective:
    def __init__(self, n_traial: int, train_loader: DataLoader, test_loader: DataLoader, n_class: int, n_epoch: int = 10):
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.n_class = n_class
        self.n_epoch = n_epoch
        self.progress_bar = tqdm(total=n_traial)

    def __call__(self, trial: optuna.trial.Trial) -> float:
        dropout_prob = trial.suggest_float("dropout_prob", 0.2, 0.8, step=0.1)
        activation = get_activation(trial)

        net = Net(self.n_class, dropout_prob, activation).to(device)
        optimizer = get_optimizer(trial, net)

        criterion = nn.CrossEntropyLoss()

        for epoch in range(self.n_epoch):
            train(net, train_loader, optimizer, criterion, verbose=False)
            loss = test(net, test_loader, criterion, verbose=False)

        self.progress_bar.update(1)

        return loss

### 自動チューニング実施(optuna)

In [17]:
n_trials = 30

n_epoch = 10
n_class = len(cifar10_classes)

In [ ]:
# 誤り率の最小化を行うため、direction='minimize'に設定
objective = Objective(n_trials, train_loader, test_loader, n_class, n_epoch)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=n_trials)

  0%|          | 0/30 [00:00<?, ?it/s]

[I 2025-11-07 00:48:06,117] A new study created in memory with name: no-name-67c2f163-b6ba-4e8a-b61c-d66330f47eed


In [ ]:
study.best_value

In [ ]:
study.best_params

In [ ]:
df = study.trials_dataframe() # pandasのDataFrame形式
df.sort_values('value')